# K562 bs128 + 早停训练与插补质量评价（本地版 / 不使用 SLURM）

与 `k562_bs128_earlystop_and_eval.ipynb`（SLURM 集群版）对应，本 notebook 是**不依赖 SLURM** 的普通版本：

1. **训练**：直接在 notebook 内用 `subprocess` 调用 `main.py`，在当前机器（有 NVIDIA GPU）上按
   **batch_size=128 + EarlyStopping(patience=25)** 依次/并行训练 K562 HiCImputeData 的 12 个条件，
   训练完自动 test 推理并写出 `denoise_recon_inv.npz`。
2. **插补质量评价**：用 GT / observed / 预测计算逐细胞 **PCC / MAE / SCC**（`all / obs / held` 子集），
   汇总成表并画图，可与官方 pipeline 结果核对。

### 使用前提

- 当前 kernel 的 Python 环境需包含本仓库依赖（torch + pytorch_lightning + anndata …），
  即训练脚本 `main.py` 能在该 Python 下运行；建议 `scdiff2` 环境。
- 机器需有可用 GPU（notebook 会先检查 `torch.cuda.is_available()`）。
- 把下面配置区的数据路径改成本机实际路径（默认使用项目内相对路径：`5_baseline/0_gtData` 与 `5_baseline/7_scHiCDiff/1_HiCImputeData/input`）。

### 生效的超参覆盖（与集群版一致）

| 覆盖项 | 值 | 作用 |
|---|---|---|
| `data.params.batch_size` | 128 | bs128 训练 |
| `data.params.test_batch_size` | 9999 | 测试集一次前向 |
| `lightning.callbacks.early_stopping_callback.params.patience` | 25 | 早停（官方协议） |
| `lightning.modelcheckpoint.params.every_n_epochs` | 50 | checkpoint 降频 |
| `--save_path` | `results/training_results_v5fast_bs128/{dataset}_sim` | 插补结果目录 |


In [ ]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import os
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import load_npz
from scipy.stats import spearmanr

# ================= 仓库定位（notebook 放在 <repo>/examples/ 下） =================
_here = Path.cwd()
if (_here / "main.py").exists():
    REPO = _here
elif (_here.parent / "main.py").exists():
    REPO = _here.parent
else:
    REPO = next((p for p in _here.parents if (p / "main.py").exists()), _here.parent)
REPO = REPO.resolve()

# ================= 数据路径（项目内相对路径，可用环境变量覆盖） =================
INPUT_DIR = Path(os.environ.get(
    "K562_INPUT_DIR",
    REPO / "5_baseline/7_scHiCDiff/1_HiCImputeData/input"))
GT_DIR = Path(os.environ.get(
    "K562_GT_DIR",
    REPO / "5_baseline/0_gtData/1_Gt_HiCImputeData"))
OBS_DIR = Path(os.environ.get(
    "K562_OBS_DIR",
    REPO / "5_baseline/0_gtData/0_downsampled_HiCImputeData"))

# ================= 输出路径 =================
SAVE_ROOT = REPO / "results/training_results_v5fast_bs128"       # 插补结果目录
LOG_ROOT = REPO / "logs/recon_masked_v5fast_bs128"               # 训练日志目录
EVAL_OUT = REPO / "results/metrics_v5fast_bs128_notebook"        # 指标输出目录
PIPELINE_CSV = REPO / "results/metrics_v5fast_bs128/1_HiCImputedData/HiCImputeData_PCC_MAE_SCC_metrics.csv"

DATASETS = [
    "K562_T1_1k", "K562_T1_2k", "K562_T1_4k", "K562_T1_7k",
    "K562_T2_1k", "K562_T2_2k", "K562_T2_4k", "K562_T2_7k",
    "K562_T3_1k", "K562_T3_2k", "K562_T3_4k", "K562_T3_7k",
]
N_CELLS, N_FEATURES = 100, 1830
ES_PATIENCE = 25
CKPT_EVERY = 50
MAX_PARALLEL = 1        # 同时跑几个数据集（显存足够可调大，例如 2）
FORCE_RETRAIN = False   # True = 已存在结果也重新训练

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print("REPO       =", REPO)
print("INPUT_DIR  =", INPUT_DIR)
print("GT_DIR     =", GT_DIR)
print("OBS_DIR    =", OBS_DIR)
print("SAVE_ROOT  =", SAVE_ROOT)

## 0. 环境与数据自检

训练脚本 `main.py` 需要 GPU（`lightning.trainer.accelerator=gpu`）。这里先确认：

- 当前 Python 能 import `torch`（同时确认 `main.py` 依赖可用）；
- `torch.cuda.is_available()` 为 True；
- 每个待训练数据集的 `{dataset}_sim.h5ad` 存在；
- 评价所需的 GT / observed npz 存在。

In [ ]:
import importlib

missing = []
for mod in ("torch", "pytorch_lightning", "anndata", "omegaconf"):
    try:
        m = importlib.import_module(mod)
        print(f"OK   {mod:20s} {getattr(m, '__version__', '')}")
    except Exception as e:
        missing.append(mod)
        print(f"FAIL {mod:20s} {e}")

import torch
cuda_ok = torch.cuda.is_available()
print("torch.cuda.is_available():", cuda_ok)
if cuda_ok:
    print("GPU:", torch.cuda.get_device_name(0), "| count:", torch.cuda.device_count())
else:
    print("!! 未检测到可用 GPU：main.py 强制 accelerator=gpu，训练会直接失败。")
    print("   请在带 GPU 的机器 / 交互式 GPU 作业中运行本 notebook。")
assert not missing, f"缺少依赖: {missing}"
assert cuda_ok, "需要可用 GPU 才能训练"

# 数据文件自检
need_h5ad = [ds for ds in DATASETS] if FORCE_RETRAIN else [
    ds for ds in DATASETS if not (SAVE_ROOT / f"{ds}_sim" / "denoise_recon_inv.npz").exists()]
bad = [str(INPUT_DIR / f"{ds}_sim.h5ad") for ds in need_h5ad
       if not (INPUT_DIR / f"{ds}_sim.h5ad").exists()]
bad += [str(GT_DIR / f"{ds}_true.npz") for ds in need_h5ad
        if not (GT_DIR / f"{ds}_true.npz").exists()]
bad += [str(OBS_DIR / f"{ds}_sim.npz") for ds in need_h5ad
        if not ((OBS_DIR / f"{ds}_sim.npz").exists() or (OBS_DIR / ds).is_dir())]
if bad:
    raise FileNotFoundError("缺少输入文件（请修改配置区路径）:\n" + "\n".join(bad[:10]))
print(f"需要训练的数据集 ({len(need_h5ad)}):", need_h5ad)
print("自检通过。")

## 1. 训练（本地 subprocess，无 SLURM）

- 每个数据集单独调用一次 `main.py`，日志写到自己进程的输出流，同时落盘到
  `logs/recon_masked_v5fast_bs128/local_{dataset}.log`。
- 已存在 `denoise_recon_inv.npz` 的数据集默认跳过；`FORCE_RETRAIN=True` 可强制重训。
- `MAX_PARALLEL` 控制并发数据集数量（1 = 顺序执行）。
- 训练完成后自动进入 test 阶段（1000 步扩散采样）并写出 4 个 npz。

把 `RUN_TRAINING` 改为 `True` 才会真正执行训练。

In [ ]:
def train_one(ds):
    save = SAVE_ROOT / f"{ds}_sim"
    h5ad = INPUT_DIR / f"{ds}_sim.h5ad"
    log = LOG_ROOT / f"local_{ds}.log"
    if save.joinpath("denoise_recon_inv.npz").exists() and not FORCE_RETRAIN:
        return ds, "SKIP", log

    save.mkdir(parents=True, exist_ok=True)
    LOG_ROOT.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable, "main.py",
        "-t", "True",
        "--base", "configs/recon_masked.yaml",
        "-n", f"{ds}_sim.seed10",
        "-l", str(LOG_ROOT),
        "--save_path", str(save),
        "data.params.batch_size=128",
        "data.params.test_batch_size=9999",
        "data.params.num_workers=4",
        f"lightning.callbacks.early_stopping_callback.params.patience={ES_PATIENCE}",
        f"lightning.modelcheckpoint.params.every_n_epochs={CKPT_EVERY}",
        "data.params.train.params.dataset=K562",
        f"data.params.train.params.fname={h5ad}",
        "data.params.validation.params.dataset=K562",
        f"data.params.validation.params.fname={h5ad}",
        "data.params.test.params.dataset=K562",
        f"data.params.test.params.fname={h5ad}",
    ]
    env = dict(os.environ, OMP_NUM_THREADS="4", OPENBLAS_NUM_THREADS="4",
               MKL_NUM_THREADS="4", NUMEXPR_NUM_THREADS="4")
    t0 = time.time()
    with open(log, "w") as fh:
        rc = subprocess.run(cmd, cwd=REPO, env=env, stdout=fh, stderr=subprocess.STDOUT).returncode
    return ds, ("DONE" if rc == 0 else f"FAIL(rc={rc})"), f"{time.time() - t0:.0f}s"


RUN_TRAINING = False   # <- 改为 True 开始训练
results = []
if RUN_TRAINING:
    todo = [ds for ds in DATASETS
            if FORCE_RETRAIN or not (SAVE_ROOT / f"{ds}_sim" / "denoise_recon_inv.npz").exists()]
    print(f"待训练 {len(todo)} 个数据集，MAX_PARALLEL={MAX_PARALLEL}")
    with ThreadPoolExecutor(max_workers=MAX_PARALLEL) as ex:
        for ds, state, info in ex.map(train_one, todo):
            print(f"{ds:12s} {state:10s} {info}")
            results.append((ds, state))
    print("训练结束。")
else:
    print("RUN_TRAINING=False，未执行训练（把该变量改为 True 即可）。")

## 2. 训练监控

- `metrics.csv` 的 `val/loss_MSE_ema` 即 EarlyStopping 监控指标（`main.py:565`）。
- 早停条件：连续 `patience=25` 个 epoch 改善 < `min_delta=1e-4`。
- test 推理结束后 `{dataset}_sim/denoise_recon_inv.npz` 出现即视为该数据集完成。

In [ ]:
def load_matrix(path):
    path = Path(path)
    if path.is_file():
        m = load_npz(path)
        if hasattr(m, "toarray"):
            m = m.toarray()
        return np.asarray(m, dtype=np.float64)
    if path.is_dir():
        tril_r, tril_c = np.tril_indices(61, k=-1)
        cells = []
        for i in range(1, 101):
            f = path / f"cell_{i}_chr19.txt"
            mat = np.zeros((61, 61), dtype=np.float64)
            if f.exists() and f.stat().st_size > 0:
                data = np.loadtxt(f, dtype=int)
                if data.ndim == 1 and data.size == 3:
                    mat[data[0], data[1]] = data[2]
                elif data.ndim == 2:
                    mat[data[:, 0], data[:, 1]] = data[:, 2]
            cells.append(mat[tril_r, tril_c])
        return np.asarray(cells, dtype=np.float64)
    stem = path.stem.removesuffix("_sim")
    alt_dir = path.parent / stem
    if alt_dir.is_dir():
        return load_matrix(alt_dir)
    raise FileNotFoundError(f"未找到数据文件或目录: {path}")


def safe_pearson(a, b):
    a = np.asarray(a, dtype=np.float64).ravel()
    b = np.asarray(b, dtype=np.float64).ravel()
    if a.size == 0 or b.size == 0 or np.std(a) == 0 or np.std(b) == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def safe_mae(a, b):
    a = np.asarray(a, dtype=np.float64).ravel()
    b = np.asarray(b, dtype=np.float64).ravel()
    if a.size == 0 or b.size == 0:
        return np.nan
    return float(np.mean(np.abs(a - b)))


def safe_spearman(a, b):
    a = np.asarray(a, dtype=np.float64).ravel()
    b = np.asarray(b, dtype=np.float64).ravel()
    if a.size < 2 or np.std(a) == 0 or np.std(b) == 0:
        return np.nan
    return float(spearmanr(a, b)[0])


def evaluate_one(ds):
    gt = load_matrix(GT_DIR / f"{ds}_true.npz")
    obs = load_matrix(OBS_DIR / f"{ds}_sim.npz")
    pred = load_matrix(SAVE_ROOT / f"{ds}_sim" / "denoise_recon_inv.npz")
    assert gt.shape == obs.shape == pred.shape == (N_CELLS, N_FEATURES), \
        f"shape mismatch: gt={gt.shape}, obs={obs.shape}, pred={pred.shape}"

    per_cell = {f"{m}_{s}": [] for m in ("pcc", "mae", "scc") for s in ("all", "obs", "held")}
    n_held = []
    for i in range(gt.shape[0]):
        g, o, p = gt[i], obs[i], pred[i]
        masks = {
            "all": np.ones_like(g, dtype=bool),
            "obs": o > 0,
            "held": (g > 0) & ~(o > 0),
        }
        n_held.append(int(masks["held"].sum()))
        for s, m in masks.items():
            per_cell[f"pcc_{s}"].append(safe_pearson(p[m], g[m]))
            per_cell[f"mae_{s}"].append(safe_mae(p[m], g[m]))
            per_cell[f"scc_{s}"].append(safe_spearman(p[m], g[m]))

    row = {"data_name": ds, "ctype": ds.split("_")[1], "cdepth": ds.split("_")[2]}
    for k, v in per_cell.items():
        row[f"{k}_mean"] = float(np.nanmean(v))
        row[f"{k}_std"] = float(np.nanstd(v))
    row["n_held_mean"] = float(np.mean(n_held))
    return row, per_cell


## 3. 插补质量评价

**输入**：GT（`{dataset}_true.npz`）、observed（`{dataset}_sim.npz`）、预测（`denoise_recon_inv.npz`）。

**指标定义**（与官方 `paperplots/1_pccAndMae_all/calculate_imputation_metrics.py` 一致）：

- 逐细胞 PCC / MAE / SCC，再跨细胞 `nanmean` / `nanstd`；
- 子集：`all`（全部 1830 特征）、`obs`（observed>0）、`held`（GT>0 且 observed≤0）；
- 原始数值直接计算，不做 log / clip / 归一化。

In [ ]:
def load_matrix(path):
    m = load_npz(path)
    if hasattr(m, "toarray"):
        m = m.toarray()
    return np.asarray(m, dtype=np.float64)


def safe_pearson(a, b):
    a = np.asarray(a, dtype=np.float64).ravel()
    b = np.asarray(b, dtype=np.float64).ravel()
    if a.size == 0 or b.size == 0 or np.std(a) == 0 or np.std(b) == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def safe_mae(a, b):
    a = np.asarray(a, dtype=np.float64).ravel()
    b = np.asarray(b, dtype=np.float64).ravel()
    if a.size == 0 or b.size == 0:
        return np.nan
    return float(np.mean(np.abs(a - b)))


def safe_spearman(a, b):
    a = np.asarray(a, dtype=np.float64).ravel()
    b = np.asarray(b, dtype=np.float64).ravel()
    if a.size < 2 or np.std(a) == 0 or np.std(b) == 0:
        return np.nan
    return float(spearmanr(a, b)[0])


def evaluate_one(ds):
    gt = load_matrix(GT_DIR / f"{ds}_true.npz")
    obs = load_matrix(OBS_DIR / f"{ds}_sim.npz")
    pred = load_matrix(SAVE_ROOT / f"{ds}_sim" / "denoise_recon_inv.npz")
    assert gt.shape == obs.shape == pred.shape == (N_CELLS, N_FEATURES), \
        f"shape mismatch: gt={gt.shape}, obs={obs.shape}, pred={pred.shape}"

    per_cell = {f"{m}_{s}": [] for m in ("pcc", "mae", "scc") for s in ("all", "obs", "held")}
    n_held = []
    for i in range(gt.shape[0]):
        g, o, p = gt[i], obs[i], pred[i]
        masks = {
            "all": np.ones_like(g, dtype=bool),
            "obs": o > 0,
            "held": (g > 0) & ~(o > 0),
        }
        n_held.append(int(masks["held"].sum()))
        for s, m in masks.items():
            per_cell[f"pcc_{s}"].append(safe_pearson(p[m], g[m]))
            per_cell[f"mae_{s}"].append(safe_mae(p[m], g[m]))
            per_cell[f"scc_{s}"].append(safe_spearman(p[m], g[m]))

    row = {"data_name": ds, "ctype": ds.split("_")[1], "cdepth": ds.split("_")[2]}
    for k, v in per_cell.items():
        row[f"{k}_mean"] = float(np.nanmean(v))
        row[f"{k}_std"] = float(np.nanstd(v))
    row["n_held_mean"] = float(np.mean(n_held))
    return row, per_cell


missing = [ds for ds in DATASETS if not (SAVE_ROOT / f"{ds}_sim" / "denoise_recon_inv.npz").exists()]
if missing:
    print("以下数据集尚未有插补结果，评价将跳过：", missing)
run_ds = [ds for ds in DATASETS if ds not in missing]

rows, per_cell_store = [], {}
for ds in run_ds:
    row, pc = evaluate_one(ds)
    rows.append(row)
    per_cell_store[ds] = pc
metrics_df = pd.DataFrame(rows)

EVAL_OUT.mkdir(parents=True, exist_ok=True)
if len(metrics_df):
    metrics_df.to_csv(EVAL_OUT / "HiCImputeData_PCC_MAE_SCC_metrics.csv", index=False)
    pd.set_option("display.width", 220)
    display(metrics_df[["data_name", "pcc_all_mean", "pcc_obs_mean", "pcc_held_mean",
                        "mae_all_mean", "mae_held_mean", "scc_held_mean", "n_held_mean"]].round(4))
    print("mean pcc_all = %.4f | mean mae_all = %.4f | mean pcc_held = %.4f"
          % (metrics_df.pcc_all_mean.mean(), metrics_df.mae_all_mean.mean(), metrics_df.pcc_held_mean.mean()))
    print("saved:", EVAL_OUT / "HiCImputeData_PCC_MAE_SCC_metrics.csv")

In [ ]:
if len(metrics_df) == 0:
    print("无评价结果可对比。")
elif PIPELINE_CSV.exists():
    pipe = pd.read_csv(PIPELINE_CSV)
    pipe = pipe[pipe["method"] == "scHiC-Diff"][
        ["data_name", "pcc_all_mean", "pcc_held_mean", "mae_all_mean"]]
    mine = metrics_df[["data_name", "pcc_all_mean", "pcc_held_mean", "mae_all_mean"]]
    cmp = pipe.merge(mine, on="data_name", suffixes=("_pipeline", "_notebook"))
    display(cmp.round(4))
    for col in ("pcc_all_mean", "pcc_held_mean", "mae_all_mean"):
        diff = (cmp[f"{col}_pipeline"] - cmp[f"{col}_notebook"]).abs().max()
        print(f"max |pipeline - notebook| {col}: {diff:.2e}")
else:
    print("未找到 pipeline 指标 CSV（可选）:", PIPELINE_CSV)

In [ ]:
if len(metrics_df):
    plot_df = metrics_df.set_index("data_name")
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    for ax, key, title in zip(
        axes,
        ["pcc_all_mean", "pcc_held_mean", "mae_all_mean"],
        ["PCC (all features)", "PCC (held-out)", "MAE (all features)"],
    ):
        err = plot_df[key.replace("_mean", "_std")]
        ax.bar(plot_df.index, plot_df[key], yerr=err, capsize=2, color="#4c72b0", alpha=0.9)
        ax.set_title(title)
        ax.tick_params(axis="x", rotation=90, labelsize=8)
    plt.tight_layout()
    plt.show()

    DS_DETAIL = "K562_T1_2k" if "K562_T1_2k" in per_cell_store else run_ds[0]
    pc = per_cell_store[DS_DETAIL]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for s, c in [("all", "#4c72b0"), ("obs", "#55a868"), ("held", "#c44e52")]:
        v = np.asarray(pc[f"pcc_{s}"], dtype=float)
        v = np.sort(v[~np.isnan(v)])
        axes[0].plot(np.arange(len(v)), v, marker=".", ms=3, lw=0.8, color=c, label=s)
    axes[0].set_xlabel("cell (sorted)")
    axes[0].set_ylabel("PCC")
    axes[0].set_title(f"Per-cell PCC by subset - {DS_DETAIL}")
    axes[0].legend()
    axes[0].set_ylim(0, 1.02)

    gt = load_matrix(GT_DIR / f"{DS_DETAIL}_true.npz")[0]
    obs = load_matrix(OBS_DIR / f"{DS_DETAIL}_sim.npz")[0]
    pred = load_matrix(SAVE_ROOT / f"{DS_DETAIL}_sim" / "denoise_recon_inv.npz")[0]
    held = (gt > 0) & ~(obs > 0)
    axes[1].scatter(gt[~held], pred[~held], s=6, alpha=0.3, label="other features")
    axes[1].scatter(gt[held], pred[held], s=8, alpha=0.7, color="#c44e52",
                    label="held-out (GT>0, obs=0)")
    lim = max(gt.max(), pred.max()) * 1.02
    axes[1].plot([0, lim], [0, lim], "k--", lw=0.8)
    axes[1].set_xlabel("GT (raw)")
    axes[1].set_ylabel("prediction (raw)")
    axes[1].set_title(f"Cell 0: prediction vs GT - {DS_DETAIL}")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

## 4. 说明

**输出位置**

```text
results/training_results_v5fast_bs128/{dataset}_sim/
├── raw_x.npz              # 观测输入（有缺失，无归一化）
├── denoise_recon.npz      # 归一化空间预测
├── denoise_recon_inv.npz  # ★ 反归一化插补结果（评价用）
└── denoise_target.npz     # 测试 target
logs/recon_masked_v5fast_bs128/local_{dataset}.log
results/metrics_v5fast_bs128_notebook/HiCImputeData_PCC_MAE_SCC_metrics.csv
```

**与集群版的差别**

- 集群版通过 `sbatch --array` 把 12 个数据集分发到多张 GPU；本版在同一台机器上以
  `subprocess` 顺序（或 `MAX_PARALLEL>1` 并行）执行，路径、超参覆盖、skip 逻辑完全一致。
- 无需 `sbatch`/`squeue`；日志直接写到本地文件。

**注意事项**

- 数据只有 100 cells（80 训练），bs128 与 bs1024 都是 1 step/epoch，插补结果应几乎一致。
- 负预测值不做 clip（与官方指标定义一致）。
- 官方 SLURM 指标 pipeline 仍以 `paperplots/1_pccAndMae_all/` 为准；本 notebook 的评价是等价轻量实现。